# 1. Ukrainian samples for model selection

Build prompts from taxonomy-matrix rows and compare Mamay 27B and Lapa 12B with
zero-shot and sequential prompting. Each matrix row fixes the group, category,
actor profile, association and context.

The initial comparison crossed 20 rows with four conditions. A separate balanced
50-row panel supplied the blinded human review. Saved inputs and outputs retain
their original versions. New model requests are optional.

In [ ]:
import json
import tempfile
from pathlib import Path

from IPython.display import display
from pandas import DataFrame

from fairforget.dataset_workflow import find_project, load_inputs
from fairforget.io import write_json

project = find_project()
inputs = load_inputs(project)
protocol = json.loads((project / "protocol/construction.json").read_text())
work = Path(tempfile.mkdtemp(prefix="warbias-dataset-"))

In [ ]:
from fairforget.warbias.pilot_analysis import describe_matrix, pilot_conditions

matrix_columns = [
    "matrix_item_id",
    "category_uk",
    "target_uk",
    "gender_track",
    "age_track",
    "stereotype_focus_uk",
    "context_uk",
]
display(DataFrame(inputs["taxonomy"])[["bias_type_code", "category_uk", "eligible_target_codes"]])
display(DataFrame(inputs["targets"])[["target_code", "target_uk", "gender_track"]])
display(DataFrame(inputs["pilot_initial_cases"])[matrix_columns])
matrix_summary = describe_matrix(inputs["matrix_full"])
matrix_summary

In [ ]:
initial_conditions = pilot_conditions(
    inputs["pilot_initial_cases"], inputs["pilot_initial_conditions"]
)
human_conditions = pilot_conditions(inputs["pilot_human_cases"], inputs["pilot_human_conditions"])
display(
    DataFrame(initial_conditions)
    .groupby(["candidate_id", "strategy"])
    .size()
    .rename("initial_items")
    .reset_index()
)
display(
    DataFrame(human_conditions)
    .groupby(["candidate_id", "strategy"])
    .size()
    .rename("human_panel_items")
    .reset_index()
)
display(
    DataFrame(inputs["pilot_initial_outputs"])[
        ["candidate_id", "strategy", "stereotype", "counter_stereotype", "unrelated"]
    ].head(8)
)

Zero-shot returns all three sentences in one request. Sequential prompting
produces the stereotype, counter-stereotype and unrelated sentence in order;
each later request receives the earlier text.

The preparation below uses the original initial-pilot prompts. To collect a new
run, configure a provider factory and set `RUN_PILOT = True`.

In [ ]:
from fairforget.dataset_workflow import prepare_pilot

snapshots = json.loads((project / "protocol/pilot_snapshots.json").read_text())
initial = snapshots["bundles"]["initial"]
candidates = [
    {
        "candidate_id": candidate_id,
        "provider": protocol["selected_condition"]["provider"],
        "model": model,
        "revision": None,
        "settings": {key: initial["settings"][key] for key in ("temperature", "max_tokens")},
    }
    for candidate_id, model in initial["model_ids"].items()
]
pilot_plan = prepare_pilot(
    project, inputs["pilot_initial_cases"], candidates=candidates, phase="initial"
)
write_json(work / "pilot-plan.json", pilot_plan)
display(DataFrame(candidates))

RUN_PILOT = False
PROVIDER_FACTORY = None
if RUN_PILOT:
    from fairforget.cli_support import load_factory
    from fairforget.warbias.pilot import run_pilot

    if not PROVIDER_FACTORY:
        raise ValueError("Set PROVIDER_FACTORY to module:callable")
    generated = run_pilot(
        pilot_plan, work / "pilot", provider_factory=load_factory(PROVIDER_FACTORY), execute=True
    )
result = {
    "stage": "ukrainian_pilot",
    "initial_matrix_rows": len(inputs["pilot_initial_cases"]),
    "initial_conditions": len(initial_conditions),
    "human_panel_conditions": len(human_conditions),
    "candidates": len(candidates),
    "strategies": list(pilot_plan["strategies"]),
    "execution_requested": RUN_PILOT,
}
result